# GAEZ RES02 Downloader 

This notebook provides a simple, reproducible workflow for downloading the GAEZ v5 Agro-climatic Potential Yield (RES02) raster datasets directly from the FAO public data repository.

The notebook automatically retrieves the latest dataset catalogue (README), allowing users to browse and filter available datasets by variable, crop, climate data source, historical or future period, Shared Socioeconomic Pathway (SSP), and water input management level. Based on the selected criteria, the corresponding GeoTIFF files are downloaded and organised into a structured folder hierarchy.

This workflow removes the need to manually search the data repository or construct download URLs, making it easier to access GAEZ RES02 products for analysis, modelling, and decision support.

## Imports and settings


In [1]:
from pathlib import Path
from urllib.parse import quote
import requests
import time
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------

# README file URL
README_URL = (
    "https://data.apps.fao.org/catalog/dataset/"
    "4316025e-fc69-47bc-a1af-09f140ea8c52/resource/"
    "bc7d61b2-20bc-4691-84fc-06d92cf51266/download/_readme_res02.xlsx"
)

# GAEZ v5 public Google Cloud base path
BASE_URL = "https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPSET"

OUT_DIR = Path("downloads_res02")
OUT_DIR.mkdir(exist_ok=True)

REQUEST_TIMEOUT = 30
CHUNK_SIZE = 1024 * 1024

## Load the RES02 README catalog and helper functions


In [2]:

def get_readme_source(readme_path=None):
    """
    Returns either a user-provided local README path or the FAO README URL.
    """
    if readme_path:
        return Path(readme_path)
    return README_URL


def clean_code_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()
            df.loc[df[col].isin(["nan", "None", ""]), col] = pd.NA

    return df.dropna(how="all")


def build_url(row):
    mapset_code = str(row["MAPSET_CODE"]).strip()
    download_name = str(row["FILE_NAME"]).strip() + ".tif"

    return f"{BASE_URL}/{quote(mapset_code)}/{quote(download_name)}"


def load_catalog_and_codes(readme_path=None):
    readme_source = get_readme_source(readme_path)

    catalog = pd.read_excel(readme_source, sheet_name="LIST_FILES", dtype=str)
    catalog = clean_code_columns(catalog)
    catalog = catalog.dropna(subset=["FILE_NAME"])

    catalog["DOWNLOAD_NAME"] = catalog["FILE_NAME"].astype(str) + ".tif"
    catalog["DOWNLOAD_URL"] = catalog.apply(build_url, axis=1)

    def read_code_sheet(sheet_name, code_col="CODE", label_col=None):
        df = pd.read_excel(readme_source, sheet_name=sheet_name, dtype=str)
        df = clean_code_columns(df)

        if label_col is None:
            label_col = [c for c in df.columns if c != code_col][0]

        df = df.dropna(subset=[code_col])
        return dict(zip(df[code_col].astype(str), df[label_col].astype(str)))

    code_maps = {
        "variable": read_code_sheet("CODES_VARIABLE", "CODE", "VARIABLE"),
        "period": read_code_sheet("CODES_PERIOD", "CODE", "PERIOD"),
        "climate": read_code_sheet(
            "CODES_CLIMATE_DATA_SOURCE",
            "CODE",
            "CLIMATE_DATA_SOURCE",
        ),
        "ssp": read_code_sheet("CODES_SSP", "CODE", "SSP"),
        "crop": read_code_sheet("CODES_CROP", "CODE", "Crop name"),
        "input": read_code_sheet(
            "CODES_WATER_INPUT",
            "CODE",
            "WATER CONTENT AND INPUT MANAGEMENT LEVEL",
        ),
    }

    return readme_source, catalog, code_maps


def label_options(values, label_map=None, include_all=False, all_label="All", all_value="__ALL__"):
    clean = [
        v for v in pd.Series(values).dropna().astype(str).unique().tolist()
        if v and v != "nan"
    ]
    clean = sorted(clean)

    options = []

    if include_all:
        options.append((all_label, all_value))

    for v in clean:
        label = f"{v} — {label_map.get(v, v)}" if label_map else v
        options.append((label, v))

    return options


def period_options(values, period_map):
    opts = [
        ("All HP historical periods", "__ALL_HP__"),
        ("All FP future periods", "__ALL_FP__"),
    ]
    opts.extend(label_options(values, period_map))
    return opts


def resolve_period_selection(selection, available_periods):
    available = pd.Series(available_periods).dropna().astype(str).unique().tolist()

    if selection == "__ALL_HP__":
        return [p for p in available if p.startswith("HP")]

    if selection == "__ALL_FP__":
        return [p for p in available if p.startswith("FP")]

    if selection == "__ALL__":
        return available

    return [selection]


def normalize_selection(value):
    if value in (None, "", "__ALL__"):
        return None

    if isinstance(value, (list, tuple, set)):
        value = list(value)
        return value or None

    return [value]


def filter_catalog(
    catalog,
    variable=None,
    periods=None,
    climates=None,
    ssps=None,
    crops=None,
    inputs=None,
):
    df = catalog.copy()

    filters = {
        "VARIABLE": normalize_selection(variable),
        "PERIOD": normalize_selection(periods),
        "CLIMATE_DATA_SOURCE": normalize_selection(climates),
        "SSP": normalize_selection(ssps),
        "CROP": normalize_selection(crops),
        "INPUT": normalize_selection(inputs),
    }

    for col, vals in filters.items():
        if vals is not None:
            df = df[df[col].isin(vals)]

    return df.reset_index(drop=True)


def head_exists(url, timeout=8):
    try:
        r = requests.head(url, timeout=timeout, allow_redirects=True)
        return r.status_code == 200
    except Exception:
        return False


def add_exists_column(df, enabled=False, max_checks=200):
    df = df.copy()

    if not enabled or df.empty:
        return df

    if len(df) > max_checks:
        df["EXISTS"] = pd.NA
        return df

    df["EXISTS"] = [head_exists(u) for u in df["DOWNLOAD_URL"]]

    return df


def download_row(row, out_dir=OUT_DIR, overwrite=False):
    url = row["DOWNLOAD_URL"]

    variable = str(row["VARIABLE"])
    period = str(row["PERIOD"])
    climate = str(row["CLIMATE_DATA_SOURCE"])
    ssp = str(row["SSP"])
    input_code = str(row["INPUT"])

    folder = Path(out_dir) / variable / f"{period}_{climate}_{ssp}_{input_code}"
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / row["DOWNLOAD_NAME"]

    if out_path.exists() and not overwrite:
        return out_path, "skipped_exists"

    with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
        r.raise_for_status()

        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)

    return out_path, "downloaded"


def download_rows(df, out_dir=OUT_DIR, overwrite=False, sleep_seconds=0.05):
    paths = []
    statuses = []

    for _, row in df.iterrows():
        path, status = download_row(row, out_dir=out_dir, overwrite=overwrite)
        paths.append(path)
        statuses.append(status)
        time.sleep(sleep_seconds)

    return pd.DataFrame({
        "path": [str(p) for p in paths],
        "status": statuses,
    })


# ------------------------------------------------------------------
# Load catalog
# ------------------------------------------------------------------

readme_source, catalog, code_maps = load_catalog_and_codes()

print("README source:", readme_source)
print("Catalog rows:", len(catalog))
print("Variables:", catalog["VARIABLE"].nunique())
print("Crops:", catalog["CROP"].nunique())

display(catalog.head())

README source: https://data.apps.fao.org/catalog/dataset/4316025e-fc69-47bc-a1af-09f140ea8c52/resource/bc7d61b2-20bc-4691-84fc-06d92cf51266/download/_readme_res02.xlsx
Catalog rows: 151232
Variables: 11
Crops: 112


,MAPSET_CODE,VARIABLE,PERIOD,CLIMATE_DATA_SOURCE,SSP,CROP,INPUT,FILE_NAME,DOWNLOAD_NAME,DOWNLOAD_URL
0,RES02-CBD,CBD,HP0120,AGERA5,HIST,ALFA,HILM,GAEZ-V5.RES02-CBD.HP0120.AGERA5.HIST.ALFA.HILM,GAEZ-V5.RES02-CBD.HP0120.AGERA5.HIST.ALFA.HILM...,https://storage.googleapis.com/fao-gismgr-gaez...
1,RES02-CBD,CBD,HP0120,AGERA5,HIST,ALFA,HRLM,GAEZ-V5.RES02-CBD.HP0120.AGERA5.HIST.ALFA.HRLM,GAEZ-V5.RES02-CBD.HP0120.AGERA5.HIST.ALFA.HRLM...,https://storage.googleapis.com/fao-gismgr-gaez...
2,RES02-CBD,CBD,HP0120,AGERA5,HIST,ALFA,LILM,GAEZ-V5.RES02-CBD.HP0120.AGERA5.HIST.ALFA.LILM,GAEZ-V5.RES02-CBD.HP0120.AGERA5.HIST.ALFA.LILM...,https://storage.googleapis.com/fao-gismgr-gaez...
3,RES02-CBD,CBD,HP0120,AGERA5,HIST,ALFA,LRLM,GAEZ-V5.RES02-CBD.HP0120.AGERA5.HIST.ALFA.LRLM,GAEZ-V5.RES02-CBD.HP0120.AGERA5.HIST.ALFA.LRLM...,https://storage.googleapis.com/fao-gismgr-gaez...
4,RES02-CBD,CBD,HP8100,AGERA5,HIST,ALFA,HILM,GAEZ-V5.RES02-CBD.HP8100.AGERA5.HIST.ALFA.HILM,GAEZ-V5.RES02-CBD.HP8100.AGERA5.HIST.ALFA.HILM...,https://storage.googleapis.com/fao-gismgr-gaez...


## Downloader Selector


In [3]:
# ------------------------------------------------------------------
# Widgets
# ------------------------------------------------------------------
mode_dd = widgets.ToggleButtons(
    options=[
        ("Download selected combination(s)", "selection"),
        ("Download all datasets for one variable", "all_variable"),
    ],
    description="Mode:",
    layout=widgets.Layout(width="850px"),
)

variable_dd = widgets.Dropdown(description="Variable:", layout=widgets.Layout(width="650px"))
period_dd = widgets.Dropdown(description="Period:", layout=widgets.Layout(width="650px"))
climate_dd = widgets.Dropdown(description="Climate:", layout=widgets.Layout(width="650px"))
ssp_dd = widgets.Dropdown(description="SSP:", layout=widgets.Layout(width="650px"))
input_dd = widgets.Dropdown(description="Input:", layout=widgets.Layout(width="650px"))

crop_ms = widgets.SelectMultiple(
    description="Crop(s):",
    options=[],
    rows=10,
    layout=widgets.Layout(width="650px"),
)

check_exists_cb = widgets.Checkbox(
    value=False,
    description="Check cloud file existence during compile (slower; skipped automatically for >200 rows)",
    layout=widgets.Layout(width="800px"),
)

overwrite_cb = widgets.Checkbox(value=False, description="Overwrite existing downloaded files")

compile_btn = widgets.Button(description="Compile selection", button_style="info", icon="check")
download_btn = widgets.Button(description="Download compiled files", button_style="success", icon="download")

file_ms = widgets.SelectMultiple(
    description="Compiled files:",
    options=[],
    rows=12,
    layout=widgets.Layout(width="1050px"),
)

download_scope = widgets.RadioButtons(
    options=[
        ("Download all compiled rows", "all_compiled"),
        ("Download only selected rows from the file list", "selected_files"),
    ],
    value="all_compiled",
    description="Download:",
    layout=widgets.Layout(width="650px"),
)

status_out = widgets.Output()
result_out = widgets.Output()
ui_out = widgets.Output()

compiled_df = pd.DataFrame()


def set_dropdown_value(dropdown, preferred=None):
    values = [v for _, v in dropdown.options]
    if preferred in values:
        dropdown.value = preferred
    elif dropdown.value in values:
        return
    elif values:
        dropdown.value = values[0]
    else:
        dropdown.value = None


def current_periods(base_df):
    return resolve_period_selection(period_dd.value, base_df["PERIOD"])


def update_dropdowns(*args):
    old_values = {
        "variable": variable_dd.value,
        "period": period_dd.value,
        "climate": climate_dd.value,
        "ssp": ssp_dd.value,
        "input": input_dd.value,
        "crops": tuple(crop_ms.value),
    }

    variable_dd.options = label_options(catalog["VARIABLE"], code_maps["variable"])
    set_dropdown_value(variable_dd, old_values["variable"])

    df_var = filter_catalog(catalog, variable=variable_dd.value)
    period_dd.options = period_options(df_var["PERIOD"], code_maps["period"])
    set_dropdown_value(period_dd, old_values["period"] or "HP0120")

    periods = current_periods(df_var)
    df_period = filter_catalog(df_var, periods=periods)

    climate_dd.options = label_options(
        df_period["CLIMATE_DATA_SOURCE"], code_maps["climate"],
        include_all=True, all_label="All climate data sources"
    )
    set_dropdown_value(climate_dd, old_values["climate"] or "__ALL__")

    climates = None if climate_dd.value == "__ALL__" else [climate_dd.value]
    df_climate = filter_catalog(df_period, climates=climates)

    ssp_dd.options = label_options(
        df_climate["SSP"], code_maps["ssp"],
        include_all=True, all_label="All SSPs"
    )
    set_dropdown_value(ssp_dd, old_values["ssp"] or "__ALL__")

    ssps = None if ssp_dd.value == "__ALL__" else [ssp_dd.value]
    df_ssp = filter_catalog(df_climate, ssps=ssps)

    input_dd.options = label_options(
        df_ssp["INPUT"], code_maps["input"],
        include_all=True, all_label="All input levels"
    )
    set_dropdown_value(input_dd, old_values["input"] or "__ALL__")

    inputs = None if input_dd.value == "__ALL__" else [input_dd.value]
    df_input = filter_catalog(df_ssp, inputs=inputs)

    crop_ms.options = label_options(df_input["CROP"], code_maps["crop"])
    available_crops = [v for _, v in crop_ms.options]
    kept = tuple([c for c in old_values["crops"] if c in available_crops])
    crop_ms.value = kept[:3]


def update_mode_ui(*args):
    with ui_out:
        clear_output()
        common_filters = widgets.VBox([
            mode_dd,
            variable_dd,
            widgets.HBox([period_dd]),
            widgets.HBox([climate_dd]),
            widgets.HBox([ssp_dd]),
            widgets.HBox([input_dd]),
            check_exists_cb,
            widgets.HBox([compile_btn, download_btn]),
            overwrite_cb,
            download_scope,
            file_ms,
            status_out,
            result_out,
        ])

        if mode_dd.value == "selection":
            display(widgets.VBox([
                mode_dd,
                variable_dd,
                widgets.HBox([period_dd]),
                widgets.HBox([climate_dd]),
                widgets.HBox([ssp_dd]),
                widgets.HBox([input_dd]),
                crop_ms,
                widgets.HTML("<b>Note:</b> select 1–3 crops for this mode."),
                check_exists_cb,
                widgets.HBox([compile_btn, download_btn]),
                overwrite_cb,
                download_scope,
                file_ms,
                status_out,
                result_out,
            ]))
        else:
            display(common_filters)


def build_current_matches(show_messages=True):
    periods = resolve_period_selection(period_dd.value, filter_catalog(catalog, variable=variable_dd.value)["PERIOD"])
    climates = None if climate_dd.value == "__ALL__" else [climate_dd.value]
    ssps = None if ssp_dd.value == "__ALL__" else [ssp_dd.value]
    inputs = None if input_dd.value == "__ALL__" else [input_dd.value]

    crops = None
    if mode_dd.value == "selection":
        crops = list(crop_ms.value)
        if len(crops) < 1 or len(crops) > 3:
            raise ValueError("Please select between 1 and 3 crops for selected-combination mode.")

    matches = filter_catalog(
        catalog,
        variable=variable_dd.value,
        periods=periods,
        climates=climates,
        ssps=ssps,
        crops=crops,
        inputs=inputs,
    )
    matches = add_exists_column(matches, enabled=check_exists_cb.value)
    return matches


def on_compile_clicked(b):
    global compiled_df
    with status_out:
        clear_output()
        print("Compiling selection...")

    with result_out:
        clear_output()
        try:
            compiled_df = build_current_matches()
        except Exception as e:
            compiled_df = pd.DataFrame()
            file_ms.options = []
            with status_out:
                clear_output()
                print("Compile error:", e)
            return

        if compiled_df.empty:
            file_ms.options = []
            print("No matching files found for the selected filters.")
            with status_out:
                clear_output()
                print("No matches.")
            return

        show_cols = [
            "MAPSET_CODE", "VARIABLE", "PERIOD", "CLIMATE_DATA_SOURCE", "SSP", "CROP", "INPUT",
            "DOWNLOAD_NAME", "DOWNLOAD_URL"
        ]
        if "EXISTS" in compiled_df.columns:
            show_cols.append("EXISTS")
        display(compiled_df[show_cols])

        file_options = [(name, name) for name in compiled_df["DOWNLOAD_NAME"].tolist()]
        file_ms.options = file_options
        file_ms.value = tuple([name for name, _ in file_options[:min(20, len(file_options))]])

    with status_out:
        clear_output()
        print(f"Compiled {len(compiled_df)} file(s). Review the table, then download.")


def on_download_clicked(b):
    global compiled_df
    with status_out:
        clear_output()

        if compiled_df.empty:
            print("No compiled files yet. Click 'Compile selection' first.")
            return

        to_download = compiled_df.copy()
        if download_scope.value == "selected_files":
            chosen = list(file_ms.value)
            if not chosen:
                print("No files selected in the compiled file list.")
                return
            to_download = to_download[to_download["DOWNLOAD_NAME"].isin(chosen)]

        print(f"Downloading {len(to_download)} file(s)...")

    with result_out:
        try:
            summary = download_rows(to_download, out_dir=OUT_DIR, overwrite=overwrite_cb.value)
            clear_output()
            print(f"Finished. Files saved under: {OUT_DIR.resolve()}")
            display(summary)
        except Exception as e:
            print("Download error:", e)
            print("If this is a 404 error, the file name exists in the README but may not be available at the expected cloud path.")

    with status_out:
        clear_output()
        print("Download step completed. Check the results table below.")


for dd in [variable_dd, period_dd, climate_dd, ssp_dd, input_dd]:
    dd.observe(update_dropdowns, names="value")
mode_dd.observe(update_mode_ui, names="value")
compile_btn.on_click(on_compile_clicked)
download_btn.on_click(on_download_clicked)

update_dropdowns()
update_mode_ui()
display(ui_out)


Output()